# Persistance

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import AIMessage
from langgraph.checkpoint.memory import InMemorySaver # MemorySaver is for storing memory (state) in same session, InMemorysaver stores memory (all states) post session
# InMemorySaver stores in RAM, not used in production setup, there is redis, postgres checkpoint
import re

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer clearly and directly."),
    ("human", "{input}")
])

llm_pipeline = HuggingFacePipeline.from_model_id(
    model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    task = "text-generation",
    pipeline_kwargs = dict(
        temperature = 0.5,
        max_new_tokens = 100
    )
)

llm = ChatHuggingFace(llm=llm_pipeline)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [ ]:
class JokeState(TypedDict):
  topic: str
  joke: str
  explanation: str

In [ ]:
def generate_joke(state: JokeState):
  prompt = f"Generate a joke on the topic {state['topic']}"
  response = llm.invoke(prompt).content
  return {'joke': response}

In [ ]:
def generate_explanation(state: JokeState):
  prompt = f"write an explanation for the joke - {state['joke']}"
  response = llm.invoke(prompt).content
  return {'explanation': response}

In [ ]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [ ]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic': 'pizza'}, config=config1)

Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'topic': 'pizza',
 'joke': '<|user|>\nGenerate a joke on the topic pizza</s>\n<|assistant|>\n"Today, my friend and I decided to order a pizza.\n\n"\'Alright, what do you want on your pizza?\' I asked.\n\n"\'I want cheese, pepperoni, and mushrooms,\' he replied.\n\n"\'And what about me? What do you want on yours?\'\n\n"\'I want pineapple, ham, and olives,\' he replied.\n\n"\'Hmm, I\'',
 'explanation': '<|user|>\nwrite an explanation for the joke - <|user|>\nGenerate a joke on the topic pizza</s>\n<|assistant|>\n"Today, my friend and I decided to order a pizza.\n\n"\'Alright, what do you want on your pizza?\' I asked.\n\n"\'I want cheese, pepperoni, and mushrooms,\' he replied.\n\n"\'And what about me? What do you want on yours?\'\n\n"\'I want pineapple, ham, and olives,\' he replied.\n\n"\'Hmm, I\'</s>\n<|assistant|>\nIn the joke, a pizza is ordered by two friends who both ask for the same toppings. The pizza is then ordered by the second friend who adds pineapple, ham, and olives to h

In [ ]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': '<|user|>\nGenerate a joke on the topic pizza</s>\n<|assistant|>\n"Today, my friend and I decided to order a pizza.\n\n"\'Alright, what do you want on your pizza?\' I asked.\n\n"\'I want cheese, pepperoni, and mushrooms,\' he replied.\n\n"\'And what about me? What do you want on yours?\'\n\n"\'I want pineapple, ham, and olives,\' he replied.\n\n"\'Hmm, I\'', 'explanation': '<|user|>\nwrite an explanation for the joke - <|user|>\nGenerate a joke on the topic pizza</s>\n<|assistant|>\n"Today, my friend and I decided to order a pizza.\n\n"\'Alright, what do you want on your pizza?\' I asked.\n\n"\'I want cheese, pepperoni, and mushrooms,\' he replied.\n\n"\'And what about me? What do you want on yours?\'\n\n"\'I want pineapple, ham, and olives,\' he replied.\n\n"\'Hmm, I\'</s>\n<|assistant|>\nIn the joke, a pizza is ordered by two friends who both ask for the same toppings. The pizza is then ordered by the second friend who adds pineapple, h

In [ ]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': '<|user|>\nGenerate a joke on the topic pizza</s>\n<|assistant|>\n"Today, my friend and I decided to order a pizza.\n\n"\'Alright, what do you want on your pizza?\' I asked.\n\n"\'I want cheese, pepperoni, and mushrooms,\' he replied.\n\n"\'And what about me? What do you want on yours?\'\n\n"\'I want pineapple, ham, and olives,\' he replied.\n\n"\'Hmm, I\'', 'explanation': '<|user|>\nwrite an explanation for the joke - <|user|>\nGenerate a joke on the topic pizza</s>\n<|assistant|>\n"Today, my friend and I decided to order a pizza.\n\n"\'Alright, what do you want on your pizza?\' I asked.\n\n"\'I want cheese, pepperoni, and mushrooms,\' he replied.\n\n"\'And what about me? What do you want on yours?\'\n\n"\'I want pineapple, ham, and olives,\' he replied.\n\n"\'Hmm, I\'</s>\n<|assistant|>\nIn the joke, a pizza is ordered by two friends who both ask for the same toppings. The pizza is then ordered by the second friend who adds pineapple, 

In [ ]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic': 'pasta'}, config=config2)

Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'topic': 'pasta',
 'joke': "<|user|>\nGenerate a joke on the topic pasta</s>\n<|assistant|>\nWhat do you get when you mix pasta with pesto? A delicious and nutritious meal that will make you feel like you're on a Mediterranean vacation!",
 'explanation': "<|user|>\nwrite an explanation for the joke - <|user|>\nGenerate a joke on the topic pasta</s>\n<|assistant|>\nWhat do you get when you mix pasta with pesto? A delicious and nutritious meal that will make you feel like you're on a Mediterranean vacation!</s>\n<|assistant|>\nThe joke is that when you combine pasta with pesto, you get a meal that's both delicious and nutritious. The pasta is cooked al dente, enhancing its flavor and texture, while the pesto adds a burst of flavor and healthy ingredients like basil, garlic, and pine nuts. The result is a satisfying and satisfying meal that's perfect for a lazy weekend or a quick dinner"}

In [ ]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'pasta', 'joke': "<|user|>\nGenerate a joke on the topic pasta</s>\n<|assistant|>\nWhat do you get when you mix pasta with pesto? A delicious and nutritious meal that will make you feel like you're on a Mediterranean vacation!", 'explanation': "<|user|>\nwrite an explanation for the joke - <|user|>\nGenerate a joke on the topic pasta</s>\n<|assistant|>\nWhat do you get when you mix pasta with pesto? A delicious and nutritious meal that will make you feel like you're on a Mediterranean vacation!</s>\n<|assistant|>\nThe joke is that when you combine pasta with pesto, you get a meal that's both delicious and nutritious. The pasta is cooked al dente, enhancing its flavor and texture, while the pesto adds a burst of flavor and healthy ingredients like basil, garlic, and pine nuts. The result is a satisfying and satisfying meal that's perfect for a lazy weekend or a quick dinner"}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkp

In [ ]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': '<|user|>\nGenerate a joke on the topic pizza</s>\n<|assistant|>\n"Today, my friend and I decided to order a pizza.\n\n"\'Alright, what do you want on your pizza?\' I asked.\n\n"\'I want cheese, pepperoni, and mushrooms,\' he replied.\n\n"\'And what about me? What do you want on yours?\'\n\n"\'I want pineapple, ham, and olives,\' he replied.\n\n"\'Hmm, I\'', 'explanation': '<|user|>\nwrite an explanation for the joke - <|user|>\nGenerate a joke on the topic pizza</s>\n<|assistant|>\n"Today, my friend and I decided to order a pizza.\n\n"\'Alright, what do you want on your pizza?\' I asked.\n\n"\'I want cheese, pepperoni, and mushrooms,\' he replied.\n\n"\'And what about me? What do you want on yours?\'\n\n"\'I want pineapple, ham, and olives,\' he replied.\n\n"\'Hmm, I\'</s>\n<|assistant|>\nIn the joke, a pizza is ordered by two friends who both ask for the same toppings. The pizza is then ordered by the second friend who adds pineapple, h

In [ ]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': '<|user|>\nGenerate a joke on the topic pizza</s>\n<|assistant|>\n"Today, my friend and I decided to order a pizza.\n\n"\'Alright, what do you want on your pizza?\' I asked.\n\n"\'I want cheese, pepperoni, and mushrooms,\' he replied.\n\n"\'And what about me? What do you want on yours?\'\n\n"\'I want pineapple, ham, and olives,\' he replied.\n\n"\'Hmm, I\'', 'explanation': '<|user|>\nwrite an explanation for the joke - <|user|>\nGenerate a joke on the topic pizza</s>\n<|assistant|>\n"Today, my friend and I decided to order a pizza.\n\n"\'Alright, what do you want on your pizza?\' I asked.\n\n"\'I want cheese, pepperoni, and mushrooms,\' he replied.\n\n"\'And what about me? What do you want on yours?\'\n\n"\'I want pineapple, ham, and olives,\' he replied.\n\n"\'Hmm, I\'</s>\n<|assistant|>\nIn the joke, a pizza is ordered by two friends who both ask for the same toppings. The pizza is then ordered by the second friend who adds pineapple, 

In [ ]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': "<|user|>\nGenerate a joke on the topic pasta</s>\n<|assistant|>\nWhat do you get when you mix pasta with pesto? A delicious and nutritious meal that will make you feel like you're on a Mediterranean vacation!", 'explanation': "<|user|>\nwrite an explanation for the joke - <|user|>\nGenerate a joke on the topic pasta</s>\n<|assistant|>\nWhat do you get when you mix pasta with pesto? A delicious and nutritious meal that will make you feel like you're on a Mediterranean vacation!</s>\n<|assistant|>\nThe joke is that when you combine pasta with pesto, you get a meal that's both delicious and nutritious. The pasta is cooked al dente, enhancing its flavor and texture, while the pesto adds a burst of flavor and healthy ingredients like basil, garlic, and pine nuts. The result is a satisfying and satisfying meal that's perfect for a lazy weekend or a quick dinner"}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'check